# CESDM schema validation

Interactive walkthrough of `model.validate()`: required relations, enums, min/max, wrong relation targets, and assign-time unit errors.

The code lives in `examples/example_validation.py` so the script and this notebook stay in sync.

After each intentional failure, the cell shows the **relevant schema (or analysis-profile) YAML excerpt** that explains why the check fails.

**Run Jupyter from the repository root** (or use the setup cell below, which `chdir`s there) so schemas and libraries resolve.


## Setup


In [1]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "schemas" / "cesdm").exists():
    # Notebook opened from notebooks/
    REPO_ROOT = REPO_ROOT.parent

# Bare names like validate_for_analysis("optimal_dispatch") look up
# analysis_profiles/ relative to CWD — keep the kernel at the repo root.
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "examples"))

from example_validation import (
    build_minimal_valid_model,
    demo_analysis_contrast,
    demo_bus_type_enum,
    demo_clean_validate,
    demo_enum_constraint,
    demo_missing_required_relation,
    demo_numeric_min_max,
    demo_range_state_of_charge,
    demo_unit_error_at_assign,
    demo_validate_or_raise,
    demo_wrong_relation_target,
    run_all_demos,
    show_schema_reason,
)

print("Repo:", REPO_ROOT)
print("CWD: ", Path.cwd())


Repo: /Users/demirayt/Downloads/sweet-cosi-cesdm
CWD:  /Users/demirayt/Downloads/sweet-cosi-cesdm


## 1. Clean baseline

`model.validate()` returns `list[str]`. Empty means structurally valid against the loaded schemas.


In [2]:
model = build_minimal_valid_model()
errors = demo_clean_validate(model)
assert errors == []



1. Clean model — model.validate()
A structurally complete CESDM model should report 0 schema errors.
  (no errors)


## 2. Missing required relation

Every `NetworkNode` needs `belongsToCarrierDomain`. The YAML excerpt below comes from the entity schema and the global relations registry.


In [3]:
errors = demo_missing_required_relation()
assert any("belongsToCarrierDomain" in e for e in errors)



2. Missing required relation
NetworkNode subclasses require belongsToCarrierDomain.
  1 error(s):
   - [ElectricalBus:bus.demo.orphan] Missing required relation 'belongsToCarrierDomain'


**Why validation fails** — `schemas/cesdm/entities/SemanticEntity/NetworkNode/NetworkNode.yaml`  
*NetworkNode declares the slot as required*

```yaml
- id: belongsToCarrierDomain
  required: true
  target: CarrierDomain
abstract: true
```

**Why validation fails** — `schemas/cesdm/relations/relations.yaml`  
*Global relation: required + target CarrierDomain*

```yaml
  belongsToCarrierDomain:
    description: >
      Relates a NetworkNode to the CarrierDomain within which its carrier exchanges
      occur and the balance constraint is enforced. Required: every node belongs to
      exactly one CarrierDomain.
    required: true
    target:
    - CarrierDomain
    cardinality: 1
```

## 3. Enum constraint

Invalid enums often **warn on assign** but still store the value. Always call `validate()`. The attribute definition in `attributes.yaml` lists the allowed values.


In [4]:
errors = demo_enum_constraint()
assert any("dispatch_type" in e and "steerable" in e for e in errors)



3. Enum constraint (dispatch_type)
Allowed: dispatchable | nondispatchable | must_run
Assigning invalid value 'steerable' (warning may print now)…
[GenerationUnit:gen.demo.wind] Value 'steerable' is not allowed for 'dispatch_type'. Allowed: ['dispatchable', 'nondispatchable', 'must_run']
  1 error(s):
   - [GenerationUnit:gen.demo.wind] Attribute 'dispatch_type' not in enum ['dispatchable', 'nondispatchable', 'must_run']: steerable


**Why validation fails** — `schemas/cesdm/attributes/attributes.yaml`  
*Allowed dispatch_type values*

```yaml
  dispatch_type:
    description: "Dispatch classification of a generation technology:\n  \"dispatchable\"    — operator\
      \ can choose output (thermal, hydro reservoir)\n  \"nondispatchable\" — output bounded by external\
      \ resource (wind, solar, RoR)\n  \"must_run\"        — must generate at minimum level (nuclear baseload,\
      \ CHP)\n"
    label: Dispatch Type
    provenance_ref: Define the provenance here.
    value:
      type: string
      constraints:
        enum:
        - dispatchable
        - nondispatchable
        - must_run
```


Fixing to 'nondispatchable'…
After fix: 0 error(s)


## 4. Numeric min / max

`nominal_power_capacity` has `minimum: 0`; `energy_conversion_efficiency` is capped at `1.0` in the attribute schema.


In [5]:
errors = demo_numeric_min_max()
assert any("minimum" in e for e in errors)
assert any("maximum" in e for e in errors)



4. Numeric min / max constraints
Setting nominal_power_capacity = -10 MW (min 0)…
[GenerationUnit:gen.demo.wind] Value -10.0 for 'nominal_power_capacity' is below minimum 0.0.
Setting energy_conversion_efficiency = 1.4 (max 1.0)…
[GenerationUnit:gen.demo.wind] Value 1.4 for 'energy_conversion_efficiency' is above maximum 1.0.
  2 error(s):
   - [GenerationUnit:gen.demo.wind] Attribute 'nominal_power_capacity' violates minimum 0.0: -10.0
   - [GenerationUnit:gen.demo.wind] Attribute 'energy_conversion_efficiency' violates maximum 1.0: 1.4


**Why validation fails** — `schemas/cesdm/attributes/attributes.yaml`  
*nominal_power_capacity ≥ 0*

```yaml
  nominal_power_capacity:
    description: Maximum instantaneous power that the conversion unit can deliver or absorb, in MW. Defines
      operational limits for dispatch.
    label: Nominal Power Capacity
    provenance_ref: Define the provenance here.
    value:
      type: decimal
      constraints:
        minimum: 0.0
    unit:
      constraints:
        enum:
        - MW
```

**Why validation fails** — `schemas/cesdm/attributes/attributes.yaml`  
*energy_conversion_efficiency in [0, 1]*

```yaml
  energy_conversion_efficiency:
    description: Ratio of useful output energy to input energy, expressed as a fraction (0–1). Determines
      efficiency of energy conversion processes.
    label: Energy Conversion Efficiency
    provenance_ref: Define the provenance here.
    value:
      type: decimal
      constraints:
        minimum: 0.0
        maximum: 1.0
    unit:
      constraints:
        enum:
        - fraction
```


Restoring valid values…
After fix: 0 error(s)


## 5. Range (state of charge)

`initial_state_of_charge` is a fraction with `minimum: 0.0` and `maximum: 1.0`.


In [6]:
errors = demo_range_state_of_charge()
assert any("initial_state_of_charge" in e for e in errors)


[StorageUnit:stor.demo.battery] Value 1.5 for 'initial_state_of_charge' is above maximum 1.0.

5. Range constraint (state of charge)
initial_state_of_charge must be between 0 and 1 inclusive.
  1 error(s):
   - [StorageUnit:stor.demo.battery] Attribute 'initial_state_of_charge' violates maximum 1.0: 1.5


**Why validation fails** — `schemas/cesdm/attributes/attributes.yaml`  
*initial_state_of_charge in [0, 1]*

```yaml
  initial_state_of_charge:
    description: Initial state of charge at the beginning of the optimization horizon, expressed as a
      fraction between 0 and 1.
    provenance_ref: Define the provenance here.
    value:
      type: decimal
      constraints:
        minimum: 0.0
        maximum: 1.0
    unit:
      constraints:
        enum:
        - fraction
```

## 6. Wrong relation target class

`hasOutputCarrier` may only point at a `Carrier` — see `target:` in `relations.yaml`.


In [7]:
errors = demo_wrong_relation_target()
assert any("hasOutputCarrier" in e and "StorageType" in e for e in errors)



6. Wrong relation target class
hasOutputCarrier expects Carrier; StorageType is incompatible.
  1 error(s):
   - [GenerationUnit:gen.demo.wind] Relation 'hasOutputCarrier' with 'Storage.Electrochemical.Battery' is of class 'StorageType' not compatible with any of [Carrier]


**Why validation fails** — `schemas/cesdm/relations/relations.yaml`  
*hasOutputCarrier may only target Carrier*

```yaml
  hasOutputCarrier:
    description: 'Relates a conversion or generation asset to the Carrier produced or injected as output.

      '
    target:
    - Carrier
```

## 7. Unit mismatch (raises immediately)

Wrong **units** raise `ValueError` at assign time — they never sit in the model for `validate()` to find. The attribute’s `unit.constraints.enum` is the source of truth.


In [8]:
demo_unit_error_at_assign()



7. Unit mismatch (assign-time ValueError)
annual_energy_demand allows only MWh/year — not GWh.
  Raised: [DemandUnit:dem.demo.elec] Unit 'GWh' is not allowed for 'annual_energy_demand'. Allowed: ['MWh/year']
  model.validate() still clean: True


**Why validation fails** — `schemas/cesdm/attributes/attributes.yaml`  
*annual_energy_demand unit enum*

```yaml
  annual_energy_demand:
    description: The total amount of energy required by the load over an entire year, expressed in MWh/year.
      It is used to scale demand time series or to validate consumption totals for modelling scenarios.
    label: Annual Energy Demand
    provenance_ref: Define the provenance here.
    value:
      type: decimal
      constraints:
        minimum: 0.0
    unit:
      constraints:
        enum:
        - MWh/year
```

## 8. Bus type enum

`powerflow_bus_type` allows only `slack`, `PV`, and `PQ`.


In [9]:
errors = demo_bus_type_enum()
assert any("powerflow_bus_type" in e for e in errors)


[ElectricalBus:bus.demo.elec] Value 'slackk' is not allowed for 'powerflow_bus_type'. Allowed: ['slack', 'PV', 'PQ']

8. Enum on ElectricalBus (powerflow_bus_type)
Allowed: slack | PV | PQ
  1 error(s):
   - [ElectricalBus:bus.demo.elec] Attribute 'powerflow_bus_type' not in enum ['slack', 'PV', 'PQ']: slackk


**Why validation fails** — `schemas/cesdm/attributes/attributes.yaml`  
*Allowed powerflow_bus_type values*

```yaml
  powerflow_bus_type:
    label: Power Flow Bus Type
    description: AC power-flow bus classification (PQ, PV, slack, …) used by the load-flow formulation.
    provenance_ref: 'IEEE Std 399-1997 (Brown Book), Section 7; MATPOWER User''s Manual, Table B-1; PSS/E
      Program Operation Manual, Bus Data.

      '
    value:
      type: string
      constraints:
        enum:
        - slack
        - PV
        - PQ
```

## 9. Schema vs analysis validation

Schema-valid models can still fail `validate_for_analysis("optimal_dispatch")`. Here the excerpt is from the **analysis profile** (not the CESDM schema) — `DemandUnit` must declare `hasDemandProfile`.

See also `examples/example_analysis_validation.py`.


In [10]:
schema_errors, analysis_errors = demo_analysis_contrast()
assert schema_errors == []
assert len(analysis_errors) >= 1



9. Schema vs analysis validation
Same model, two questions:
  model.validate()                        → 0 error(s)
  validate_for_analysis('optimal_dispatch.yaml') → 1 error(s)
  Sample analysis findings:
   - [optimal_dispatch] [DemandUnit:dem.demo.elec] missing required 'hasDemandProfile'

  Tip: see examples/example_analysis_validation.py for the full study-readiness walkthrough.


**Why validation fails** — `analysis_profiles/optimal_dispatch.yaml`  
*Study profile (not schema): DemandUnit needs hasDemandProfile*

```yaml
  - entity_class: DemandUnit
    checks:
      - attribute: atNode
        required: true

      - attribute: annual_energy_demand
        required: true
        # The energy budget a dispatch study distributes over time,
        # together with hasDemandProfile below -- not
        # maximum_energy_demand, which only bounds the peak and says
        # nothing about the total energy actually consumed.
        constraints:
          minimum: 0

      - attribute: hasDemandProfile
        required: true
        # The temporal shape annual_energy_demand is distributed over;
        # without it there's no way to know which hours the energy is
        # actually needed in.

      - attribute: maximum_energy_demand
        required: false
        constraints:
          minimum: 0

      - attribute: value_of_lost_load
        required: false
        # Only meaningful if the study allows curtailing demand at a
        # cost rather than treating it as a hard constraint.
        constraints:
          minimum: 0
```

## 10. `validate_or_raise()`

Same enum constraint as section 3; this helper raises instead of returning a list.


In [11]:
demo_validate_or_raise()


[GenerationUnit:gen.demo.wind] Value 'steerable' is not allowed for 'dispatch_type'. Allowed: ['dispatchable', 'nondispatchable', 'must_run']

10. validate_or_raise()
  Raised ValueError (160 chars). First line:
   CESDM model validation failed:


**Why validation fails** — `schemas/cesdm/attributes/attributes.yaml`  
*Same enum constraint as section 3*

```yaml
  dispatch_type:
    description: "Dispatch classification of a generation technology:\n  \"dispatchable\"    — operator\
      \ can choose output (thermal, hydro reservoir)\n  \"nondispatchable\" — output bounded by external\
      \ resource (wind, solar, RoR)\n  \"must_run\"        — must generate at minimum level (nuclear baseload,\
      \ CHP)\n"
    label: Dispatch Type
    provenance_ref: Define the provenance here.
    value:
      type: string
      constraints:
        enum:
        - dispatchable
        - nondispatchable
        - must_run
```

## Optional: run everything at once


In [12]:
# Uncomment to replay the full CLI narrative:
# run_all_demos()
